# 03 — Preparação do Dataset: Somente Dados do Cliente

> **Dados do cliente são mantidos isolados** — nunca misturados com dados públicos.

Consome os segmentos rotulados de `data/client_labeled/` e gera um dataset YOLO  
completo em `data/datasets/client_only/`, pronto para o treinamento do notebook 05.

**Classes:** `0=person` · `1=violence` · `2=pre_violence`

In [ ]:
!pip install -q ultralytics opencv-python-headless mediapipe scikit-learn tqdm PyYAML matplotlib

In [ ]:
import cv2, yaml, json, shutil
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from IPython.display import display

mp_pose = mp.solutions.pose

LABELED_DIR  = Path("../data/client_labeled")        # saída do nb02
DATASET_DIR  = Path("../data/datasets/client_only")  # dataset YOLO isolado

EXTRACT_FPS  = 5
IMG_SIZE     = 640
PRE_V_THRESH = 0.45
CLASS_NAMES  = ["person", "violence", "pre_violence"]

for split in ["train","val","test"]:
    (DATASET_DIR/"images"/split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR/"labels"/split).mkdir(parents=True, exist_ok=True)
(DATASET_DIR/"metadata").mkdir(exist_ok=True)

assert LABELED_DIR.exists(), "Execute 02_client_classification.ipynb primeiro."
print(f"Origem  : {LABELED_DIR.resolve()}")
print(f"Dataset : {DATASET_DIR.resolve()}")

In [ ]:
# Funções reutilizadas dos notebooks anteriores
_yolo_person = YOLO("yolov8n.pt")

def detect_bboxes(frame):
    res = _yolo_person(frame, classes=[0], verbose=False)[0]
    boxes = [tuple(int(v) for v in b) for b in res.boxes.xyxy.tolist()]
    return boxes or [(0,0,frame.shape[1],frame.shape[0])]

def pose_score(frame):
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.4) as p:
        res = p.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if not res.pose_landmarks: return 0.0
        lm = res.pose_landmarks.landmark
        def pt(i): return np.array([lm[i].x, lm[i].y])
        try:
            shy  = (pt(11)[1]+pt(12)[1])/2
            arm  = min(1.0,(max(0,shy-pt(15)[1])+max(0,shy-pt(16)[1]))*2)
            tor  = min(1.0,abs(np.arctan2((pt(0)-(pt(23)+pt(24))/2)[0],
                                          -(pt(0)-(pt(23)+pt(24))/2)[1]))/(np.pi/3))
            spd  = min(1.0,np.linalg.norm(pt(15)-pt(16))*2)
            return float(np.clip(0.4*arm+0.3*tor+0.3*spd,0,1))
        except: return 0.0

def to_yolo(w, h, bbox, cls):
    x1,y1,x2,y2 = bbox
    return f"{cls} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}"

def process_video(path, label):
    cap  = cv2.VideoCapture(str(path))
    if not cap.isOpened(): return []
    fps  = cap.get(cv2.CAP_PROP_FPS) or 25
    skip = max(1, int(fps/EXTRACT_FPS))
    recs, fi = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if fi % skip == 0:
            h,w = frame.shape[:2]
            ps  = pose_score(frame)
            if label == "violence":          cls = 1
            elif ps >= PRE_V_THRESH:         cls = 2
            else:                            cls = 0
            bboxes = detect_bboxes(frame)
            recs.append(dict(frame=frame, class_id=cls, pose_score=ps,
                             annots=[to_yolo(w,h,b,cls) for b in bboxes],
                             source=path.name))
        fi += 1
    cap.release()
    return recs

In [ ]:
all_recs = []
for label in ["violence", "non_violence"]:
    vids = list((LABELED_DIR/label).glob("*.mp4")) + list((LABELED_DIR/label).glob("*.avi"))
    for v in tqdm(vids, desc=label):
        all_recs.extend(process_video(v, label))

print(f"\nFrames totais : {len(all_recs)}")
for c,n in enumerate(CLASS_NAMES):
    print(f"  {n:14s}: {sum(1 for r in all_recs if r['class_id']==c)}")

In [ ]:
# Split estratificado
labels = [r["class_id"] for r in all_recs]
tr_i,tmp_i = train_test_split(range(len(all_recs)),test_size=0.3,stratify=labels,random_state=42)
tmp_l = [labels[i] for i in tmp_i]
vl_i,te_i  = train_test_split(tmp_i, test_size=0.33, stratify=tmp_l, random_state=42)

for split, idxs in [("train",tr_i),("val",vl_i),("test",te_i)]:
    img_d = DATASET_DIR/"images"/split
    lbl_d = DATASET_DIR/"labels"/split
    for k,i in enumerate(tqdm(idxs, desc=split)):
        rec   = all_recs[i]
        frame = cv2.resize(rec["frame"], (IMG_SIZE, IMG_SIZE))
        name  = f"{split}_{k:06d}"
        cv2.imwrite(str(img_d/f"{name}.jpg"), frame)
        (lbl_d/f"{name}.txt").write_text("\n".join(rec["annots"]))

# dataset.yaml
ds_yaml = dict(path=str(DATASET_DIR.resolve()),
               train="images/train", val="images/val", test="images/test",
               nc=3, names=CLASS_NAMES)
yaml_path = DATASET_DIR/"dataset.yaml"
with open(yaml_path,"w") as f: yaml.dump(ds_yaml, f, default_flow_style=False)

# Stats
stats = dict(source="client_only", total=len(all_recs),
             split=dict(train=len(tr_i),val=len(vl_i),test=len(te_i)),
             classes={n:sum(1 for r in all_recs if r["class_id"]==i)
                      for i,n in enumerate(CLASS_NAMES)})
with open(DATASET_DIR/"metadata"/"stats.json","w") as f: json.dump(stats,f,indent=2)

print(f"\n✓ Dataset client_only pronto")
print(f"  train={len(tr_i)}  val={len(vl_i)}  test={len(te_i)}")
print(f"  YAML: {yaml_path}")
print("\nPróximo: 05_train_client_only.ipynb")

In [ ]:
# Visualização
fig, axes = plt.subplots(1,2,figsize=(12,4))
colors=["steelblue","tomato","darkorange"]
axes[0].bar(CLASS_NAMES,[stats["classes"][n] for n in CLASS_NAMES],color=colors,edgecolor="black",alpha=0.85)
axes[0].set_title("Distribuição de Classes — client_only")
axes[1].bar(["train","val","test"],[stats["split"][s] for s in ["train","val","test"]],
             color="slateblue",edgecolor="black",alpha=0.85)
axes[1].set_title("Split")
plt.suptitle("Dataset: Somente Dados do Cliente",fontsize=13)
plt.tight_layout()
plt.show()